# 11 · Modelado predictivo

**Proyecto Integrador · Tecnicatura en Ciencia de Datos e Inteligencia Artificial**

Cierre del pipeline: sobre el dataset de variables que dejó el notebook 04
(`data/pipeline/02_df_features.csv`), entrenamos los **dos modelos** del proyecto.

| # | Modelo | Tipo | Pregunta de negocio |
|---|---|---|---|
| 1 | **Clasificación de reseña** | Supervisado (binario) | ¿Esta compra va a terminar en reseña **positiva** o **negativa**, sabiéndolo *al momento de comprar*? |
| 2 | **Segmentación de vendedores** | No supervisado (K-Means) | ¿En qué **perfiles** se agrupan los vendedores según su logística y su negocio? |

El modelo de clasificación se entrena, se evalúa y se **serializa** (junto con todo lo necesario
para predecir una compra nueva) en `modelos_output/`, de modo que el dashboard (`app.py`) solo
tenga que **leer** los resultados, sin reentrenar nada.

**Criterio de explicación** (el mismo del resto del pipeline): cada celda de código va precedida de
un markdown con **Qué hacemos** y **Para qué**, y seguida del **Insight** cuando corresponde.

## Celda estándar: imports + rutas + semilla

**Qué hacemos:** importamos pandas/numpy, los modelos de scikit-learn y LightGBM, las métricas y
`joblib` para serializar; fijamos las rutas del proyecto (detectando la raíz aunque Jupyter arranque
desde `notebooks/`) y una semilla única.

**Para qué:** dejar toda la configuración en un solo lugar y garantizar que el entrenamiento sea
**reproducible** (misma semilla → mismos resultados).

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# raíz del proyecto (aunque se abra desde notebooks/)
BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

PIPELINE_DIR = BASE / "data" / "pipeline"
OUT_DIR = BASE / "modelos_output"
OUT_DIR.mkdir(exist_ok=True)

SUDESTE = {"SP", "RJ", "MG", "ES"}
RANDOM_STATE = 42

pd.set_option("display.max_columns", 60)
print(f"Raíz del proyecto: {BASE}")
print(f"Salidas del modelado -> {OUT_DIR}")

Raíz del proyecto: C:\Olist_Ecommerce\Olist_Analisis
Salidas del modelado -> C:\Olist_Ecommerce\Olist_Analisis\modelos_output


## Carga del dataset de variables

**Qué hacemos:** leemos `data/pipeline/02_df_features.csv`, el artefacto que dejó el notebook 04 con
todas las variables ya calculadas (distancia, volumen, flete, demora, reseña, etc.) y reconvertimos
las columnas de fecha.

**Para qué:** el modelado parte **exactamente** del mismo dataset que el EDA, sin recalcular nada: así
las dos etapas del proyecto son coherentes entre sí.

In [2]:
ruta_features = PIPELINE_DIR / "02_df_features.csv"
if not ruta_features.exists():
    raise FileNotFoundError(
        f"No existe {ruta_features}. Ejecutá primero los notebooks 02 → 04 del pipeline."
    )

df = pd.read_csv(ruta_features)
for col in ["order_purchase_timestamp", "order_delivered_customer_date",
            "order_estimated_delivery_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Dataset de modelado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Dataset de modelado: 112,650 filas x 43 columnas


---
## Modelo 1 — Clasificación de la reseña (al momento de la compra)

**La pregunta:** ¿se puede anticipar si una compra va a terminar en reseña **positiva** (4-5 ★) o
**negativa** (1-2 ★) usando **solo lo que se conoce en el instante de comprar**?

Dos decisiones de diseño sostienen todo el modelo:

- **Target binario** (positiva vs. negativa, descartando la reseña neutral de 3 ★). La reseña neutral
  es la más ambigua —una mezcla de "conforme" y "algo decepcionado"— y mete ruido; separarla deja un
  problema más nítido y accionable.
- **Sin fuga de información:** no usamos nada que ocurra *después* de comprar (ni el texto de la
  reseña, ni la demora real del envío, ni `review_score`). Solo variables disponibles al momento de la
  compra: precio, flete, distancia, producto, medio de pago, mes, geografía y el **historial pasado
  del vendedor**. Esa es la versión útil de la pregunta: sirve para decidir a qué pedidos prestar
  atención *antes* de que la reseña ya esté escrita.

### Variables conocidas al momento de la compra

**Qué hacemos:** derivamos las variables temporales (mes, día de la semana, temporada alta) y la
geográfica `mismo_estado`, definimos el target binario `reseña_binaria` y agrupamos las columnas en
bloques (numéricas base, temporales, geográficas, categóricas).

**Para qué:** dejar explícito el conjunto de *features* y asegurarnos de que **todas** son observables
en el instante de la compra — ninguna depende del desenlace del pedido.

In [3]:
# --- temporales (se conocen al comprar) ---
df["mes_compra_num"] = df["order_purchase_timestamp"].dt.month
df["dia_semana"] = df["order_purchase_timestamp"].dt.dayofweek
df["es_temporada_alta"] = df["mes_compra_num"].isin([11, 12, 1]).astype(int)  # Black Friday, Navidad, verano

# --- geográfica (se conoce al comprar) ---
df["mismo_estado"] = (df["customer_state"] == df["seller_state"]).astype(int)

# --- target binario: 0 = negativa (1-2), 1 = positiva (4-5); la neutral (3) queda fuera ---
def target_binario(score):
    if pd.isna(score):
        return np.nan
    if score <= 2:
        return 0
    if score >= 4:
        return 1
    return np.nan

df["reseña_binaria"] = df["review_score"].apply(target_binario)

# bloques de features (todas conocidas al momento de la compra)
NUM_BASE = ["distancia_km", "volumen_cm3", "product_weight_g", "price", "freight_value",
            "flete_ratio", "items_por_pedido", "payment_installments_max"]
NUM_VENDEDOR = ["hist_pct_demora_vendedor", "hist_pedidos_vendedor",
                "hist_resena_prom_vendedor", "hist_categorias_vendedor"]
NUM_TEMPORAL = ["mes_compra_num", "dia_semana", "es_temporada_alta"]
NUM_GEO = ["mismo_estado"]
CAT = ["product_category_name_english", "region_sudeste", "payment_type_principal", "customer_state"]

reparto = df["reseña_binaria"].value_counts(dropna=False).sort_index()
print("Distribución del target (0=negativa, 1=positiva, NaN=neutral descartada):")
print(reparto.to_string())
print(f"\n% positivas (sobre no neutrales): {df['reseña_binaria'].mean(skipna=True) * 100:.1f}%")

Distribución del target (0=negativa, 1=positiva, NaN=neutral descartada):
reseña_binaria
0.0    17986
1.0    84347
NaN    10317

% positivas (sobre no neutrales): 82.4%


**Insight:** el dataset está **muy desbalanceado** hacia las reseñas positivas (la mayoría de los
pedidos de Olist se entregan bien). Por eso más adelante medimos con **F1 macro** y **AUC** —no con
accuracy— y entrenamos con `class_weight="balanced"`: queremos acertar las **negativas**, que son
pocas pero son las que importan.

### Separación train / test

**Qué hacemos:** nos quedamos con las filas que tienen target (positiva o negativa), imputamos las
numéricas base con su mediana y partimos en **train (75%)** y **test (25%)** de forma estratificada por
el target.

**Para qué:** el split estratificado conserva la proporción de clases en ambas partes, y arrastramos
`seller_id` sin usarlo como feature: lo necesitamos solo para construir el historial del vendedor en el
paso siguiente.

In [4]:
datos = df.dropna(subset=["reseña_binaria"]).copy()
y = datos["reseña_binaria"].astype(int)

cols_arrastre = (["seller_id", "seller_state", "envio_demorado", "review_score"]
                 + NUM_BASE + NUM_TEMPORAL + NUM_GEO + CAT)
X_all = datos[cols_arrastre].copy()

# imputación de numéricas base (medianas se guardan para poder predecir compras nuevas)
medianas = {c: float(X_all[c].median()) for c in NUM_BASE}
for col in NUM_BASE:
    X_all[col] = X_all[col].fillna(medianas[col])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train_raw):,} filas | Test: {len(X_test_raw):,} filas")

Train: 76,749 filas | Test: 25,584 filas


### Historial del vendedor (calculado solo con train)

**Qué hacemos:** para cada vendedor calculamos, **usando únicamente las filas de train**, cuatro
señales de su comportamiento pasado: % histórico de envíos demorados, cantidad de pedidos, reseña
promedio histórica y cantidad de categorías distintas que vende. Los vendedores sin historial (o que
solo aparecen en test) reciben el promedio global de train.

**Para qué:** el historial pasado del vendedor es información **legítima y disponible** al momento de
la compra. Calcularlo solo con train evita la **fuga de información**: nunca mira el resultado del
propio pedido que queremos predecir.

In [5]:
idx_tr = X_train_raw.index
hist_demora = (datos.loc[idx_tr, "envio_demorado"]
               .groupby(X_train_raw["seller_id"]).agg(["mean", "count"]))
hist_demora.columns = ["hist_pct_demora_vendedor", "hist_pedidos_vendedor"]
hist_resena = datos.loc[idx_tr, "review_score"].groupby(X_train_raw["seller_id"]).mean()
hist_cats = (datos.loc[idx_tr, "product_category_name_english"]
             .groupby(X_train_raw["seller_id"]).nunique())
seller_estado = datos.loc[idx_tr].groupby(X_train_raw["seller_id"])["seller_state"].first()

prom_demora_global = float(datos.loc[idx_tr, "envio_demorado"].mean())
prom_resena_global = float(datos.loc[idx_tr, "review_score"].mean())

historial = pd.DataFrame(hist_demora)
historial["hist_resena_prom_vendedor"] = hist_resena
historial["hist_categorias_vendedor"] = hist_cats
historial["seller_state"] = seller_estado


def agregar_historial(parte):
    out = parte.merge(historial.drop(columns="seller_state"),
                      left_on="seller_id", right_index=True, how="left")
    out["hist_pct_demora_vendedor"] = out["hist_pct_demora_vendedor"].fillna(prom_demora_global)
    out["hist_pedidos_vendedor"] = out["hist_pedidos_vendedor"].fillna(0)
    out["hist_resena_prom_vendedor"] = out["hist_resena_prom_vendedor"].fillna(prom_resena_global)
    out["hist_categorias_vendedor"] = out["hist_categorias_vendedor"].fillna(0)
    return out


X_train_h = agregar_historial(X_train_raw)
X_test_h = agregar_historial(X_test_raw)
print(f"Vendedores con historial real en train: {len(historial):,}")

Vendedores con historial real en train: 2,911


### Entrenamiento del clasificador

**Qué hacemos:** armamos la matriz final (one-hot de las categóricas, reindexando test con las
columnas de train) y entrenamos un **LightGBM** con `class_weight="balanced"`. LightGBM es un gradient
boosting sobre árboles: rápido, robusto a variables en distinta escala y con buen manejo de las
categóricas codificadas.

**Para qué:** LightGBM es el modelo que mejor captura las interacciones no lineales entre estas
variables tabulares, y el balanceo de clases lo obliga a no ignorar la minoría (reseñas negativas).

In [6]:
NUM_COLS = NUM_BASE + NUM_VENDEDOR + NUM_TEMPORAL + NUM_GEO
CAT_COLS = CAT


def construir_matriz(Xtr, Xte):
    Xtr = pd.get_dummies(Xtr[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=True)
    Xte = pd.get_dummies(Xte[NUM_COLS + CAT_COLS], columns=CAT_COLS, drop_first=True)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    return Xtr, Xte


X_train, X_test = construir_matriz(X_train_h, X_test_h)
COLUMNAS_MODELO = list(X_train.columns)

modelo = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=-1,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
modelo.fit(X_train, y_train)
print(f"Modelo entrenado con {X_train.shape[1]} variables.")

Modelo entrenado con 117 variables.


### Evaluación en test

**Qué hacemos:** predecimos sobre el test y calculamos accuracy, precision/recall/F1 **macro** (que
pesan por igual las dos clases) y **AUC**. Guardamos la comparación, la matriz de confusión y las
predicciones en `modelos_output/`.

**Para qué:** las métricas macro y el AUC son las honestas para un problema desbalanceado; la matriz de
confusión muestra *cómo* se reparten los aciertos y errores entre positivas y negativas.

In [7]:
pred = modelo.predict(X_test)
proba = modelo.predict_proba(X_test)[:, 1]

metricas = {
    "accuracy": accuracy_score(y_test, pred),
    "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
    "auc": roc_auc_score(y_test, proba),
}
comparacion = pd.DataFrame([{"modelo": "LightGBM (binario, al comprar)", **metricas}]).set_index("modelo")
comparacion.round(3).to_csv(OUT_DIR / "clasificacion_comparacion.csv")

etiquetas = ["Negativa", "Positiva"]
cm = confusion_matrix(y_test, pred)
pd.DataFrame(cm, index=[f"real_{e}" for e in etiquetas],
             columns=[f"pred_{e}" for e in etiquetas]).to_csv(
    OUT_DIR / "clasificacion_matriz_confusion.csv")

pd.DataFrame({"y_real": y_test.values, "prediccion": pred,
              "proba_positiva": proba}).to_csv(
    OUT_DIR / "clasificacion_predicciones_test.csv", index=False)

print(comparacion.round(3).to_string())
print("\nMatriz de confusión (filas=real, columnas=predicho):")
print(pd.DataFrame(cm, index=etiquetas, columns=etiquetas).to_string())

                                accuracy  precision_macro  recall_macro  f1_macro    auc
modelo                                                                                  
LightGBM (binario, al comprar)     0.718            0.612         0.666     0.618  0.718

Matriz de confusión (filas=real, columnas=predicho):
          Negativa  Positiva
Negativa      2635      1862
Positiva      5346     15741


**Insight:** con **solo** variables conocidas al momento de la compra el modelo alcanza un
**AUC ≈ 0,72** y un **F1 macro ≈ 0,62** — modesto pero legítimo: no hay trampa de fuga. El techo es
intrínseco al problema, porque gran parte de lo que define la reseña (cómo salió la entrega) todavía no
ocurrió cuando el cliente compra. Aun así, separa bastante mejor que el azar y sirve para **señalar
pedidos de riesgo** antes de que lleguen.

### Importancia de las variables

**Qué hacemos:** ordenamos las variables por su peso en el modelo y guardamos el top 15.

**Para qué:** entender *qué señales* usa el modelo convierte la caja negra en algo accionable para el
negocio.

In [8]:
importancias = pd.Series(modelo.feature_importances_, index=COLUMNAS_MODELO)
top = importancias.sort_values(ascending=False).head(15)
top.to_csv(OUT_DIR / "clasificacion_importancia.csv", header=["importancia"])
print(top.to_string())

hist_resena_prom_vendedor    931
distancia_km                 844
freight_value                750
mes_compra_num               658
flete_ratio                  613
price                        563
product_weight_g             535
hist_pct_demora_vendedor     529
volumen_cm3                  520
hist_pedidos_vendedor        517
items_por_pedido             446
payment_installments_max     304
dia_semana                   297
hist_categorias_vendedor     235
customer_state_RJ            147


**Insight:** la variable más influyente es **`hist_resena_prom_vendedor`** — la reseña histórica
promedio del vendedor. El mejor predictor de si un cliente va a quedar conforme es **cómo quedaron los
clientes anteriores de ese mismo vendedor**. La distancia y el costo de flete también pesan, y aparece
`mes_compra_num` entre las relevantes. La línea más prometedora para seguir mejorando es sumar más
señales de comportamiento del vendedor.

### Serialización del modelo final

**Qué hacemos:** guardamos en `modelos_output/modelo_final.joblib` un *bundle* con todo lo necesario
para predecir una compra nueva **sin reentrenar**: el modelo, las columnas exactas del one-hot, las
medianas de imputación, los promedios globales, el historial por vendedor, las opciones de las
categóricas y las métricas. Además exportamos un **catálogo de vendedores** (los que tienen historial
real) para poder elegirlos en el dashboard.

**Para qué:** el dashboard (`app.py`) carga este archivo y ofrece la predicción interactiva al instante,
sin tener que repetir todo el entrenamiento cada vez que alguien abre la app.

In [9]:
catalogo = historial.reset_index().rename(columns={"index": "seller_id"})
catalogo = catalogo[catalogo["hist_pedidos_vendedor"] > 0].copy()
catalogo = catalogo.sort_values("hist_pedidos_vendedor", ascending=False)
catalogo.to_csv(OUT_DIR / "catalogo_vendedores.csv", index=False)

opciones_categoricas = {
    c: sorted(datos[c].dropna().unique().tolist())
    for c in ["product_category_name_english", "payment_type_principal", "customer_state"]
}

bundle = {
    "modelo": modelo,
    "columnas_modelo": COLUMNAS_MODELO,
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
    "num_base": NUM_BASE,
    "medianas": medianas,
    "prom_demora_global": prom_demora_global,
    "prom_resena_global": prom_resena_global,
    "opciones_categoricas": opciones_categoricas,
    "sudeste": sorted(SUDESTE),
    "descripcion": "LightGBM binario (al momento de la compra) — reseña negativa(0) / positiva(1)",
    "metricas": {"f1_macro": metricas["f1_macro"], "auc": metricas["auc"]},
}
joblib.dump(bundle, OUT_DIR / "modelo_final.joblib")
print(f"Modelo + artefactos -> {OUT_DIR / 'modelo_final.joblib'}")
print(f"Catálogo de vendedores ({len(catalogo):,}) -> {OUT_DIR / 'catalogo_vendedores.csv'}")

Modelo + artefactos -> C:\Olist_Ecommerce\Olist_Analisis\modelos_output\modelo_final.joblib
Catálogo de vendedores (2,801) -> C:\Olist_Ecommerce\Olist_Analisis\modelos_output\catalogo_vendedores.csv


---
## Modelo 2 — Segmentación de vendedores (K-Means)

**La pregunta:** en el EDA vimos que los vendedores con peor logística no son necesariamente los más
grandes. Un **clustering** formaliza esa intuición: en vez de mirar cada métrica por separado, agrupa a
los vendedores según su **perfil completo** (logística + negocio). Usamos **K-Means**, el algoritmo de
clustering más estándar e interpretable, y elegimos la cantidad de grupos con el **método del codo** y
el **puntaje de silhouette**.

### Perfil de cada vendedor

**Qué hacemos:** agregamos el dataset a nivel de vendedor con ocho variables: cinco logísticas (tiempo
de entrega, % de demora, distancia, flete y cantidad de pedidos) y tres de negocio (reseña promedio,
precio promedio y categorías distintas). A `pedidos` y `precio_promedio` les aplicamos `log1p` antes de
escalar, y estandarizamos todo.

**Para qué:** mezclar señales logísticas y de negocio hace que los grupos separen por **calidad** y no
solo por velocidad; el `log1p` evita que un puñado de vendedores enormes domine la distancia por su
escala, y el escalado pone a todas las variables en pie de igualdad.

In [10]:
perfil = df.groupby("seller_id").agg(
    pedidos=("order_id", "nunique"),
    tiempo_entrega_promedio=("tiempo_entrega_dias", "mean"),
    pct_demorado=("envio_demorado", "mean"),
    distancia_promedio=("distancia_km", "mean"),
    flete_promedio=("freight_value", "mean"),
    reseña_promedio=("review_score", "mean"),
    precio_promedio=("price", "mean"),
    categorias_distintas=("product_category_name_english", "nunique"),
).dropna()

FEATURES_CLUSTER = ["tiempo_entrega_promedio", "pct_demorado", "distancia_promedio",
                    "flete_promedio", "pedidos", "reseña_promedio",
                    "precio_promedio", "categorias_distintas"]

X_cluster_df = perfil[FEATURES_CLUSTER].copy()
for col in ["pedidos", "precio_promedio"]:
    X_cluster_df[col] = np.log1p(X_cluster_df[col])

X_cluster = StandardScaler().fit_transform(X_cluster_df)
print(f"Vendedores a segmentar: {len(perfil):,}")

Vendedores a segmentar: 2,960


### Elección de la cantidad de clusters

**Qué hacemos:** probamos K-Means con k de 2 a 8 y, para cada k, registramos la inercia (codo) y el
silhouette.

**Para qué:** no hay una cantidad "correcta" dada de antemano; el codo muestra dónde sumar grupos deja
de reducir el error y el silhouette mide qué tan separados quedan. Entre los dos, eligen un k razonable.

In [11]:
resultados_k = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster)
    resultados_k.append({"k": k, "inercia": km.inertia_,
                         "silhouette": silhouette_score(X_cluster, labels)})

resultados_k = pd.DataFrame(resultados_k)
resultados_k.to_csv(OUT_DIR / "clustering_elbow_silhouette.csv", index=False)
print(resultados_k.round(3).to_string(index=False))

 k   inercia  silhouette
 2 20055.519       0.330
 3 17347.848       0.181
 4 14917.925       0.204
 5 13580.907       0.169
 6 12451.347       0.185
 7 11429.045       0.197
 8 10716.569       0.185


**Insight:** valores de silhouette en torno a 0,2-0,3 son **normales en datos de negocio reales**
(no forman grupos perfectamente separados como un dataset de tutorial). El codo y el silhouette apuntan
a **k = 3** como el equilibrio más interpretable.

### Perfil de los tres segmentos

**Qué hacemos:** entrenamos K-Means con k = 3, asignamos cluster a cada vendedor y resumimos el perfil
promedio de cada grupo, ordenados por % de demora. Guardamos resumen y asignaciones.

**Para qué:** convierte la salida del algoritmo en tres perfiles de negocio **nombrables** y
accionables.

In [12]:
K = 3
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
perfil["cluster"] = kmeans.fit_predict(X_cluster)

resumen = perfil.groupby("cluster")[FEATURES_CLUSTER].mean().round(1)
resumen["n_vendedores"] = perfil.groupby("cluster").size()
resumen = resumen.sort_values("pct_demorado")
resumen.to_csv(OUT_DIR / "clustering_resumen.csv")
perfil.reset_index().to_csv(OUT_DIR / "clustering_asignaciones.csv", index=False)
print(resumen.to_string())

         tiempo_entrega_promedio  pct_demorado  distancia_promedio  flete_promedio  pedidos  reseña_promedio  precio_promedio  categorias_distintas  n_vendedores
cluster                                                                                                                                                          
1                            9.5           0.0               519.6            18.7      6.6              4.2            116.4                   1.4          1704
2                           11.8           0.1               594.8            20.1    107.1              4.1            130.8                   4.2           795
0                           19.6           0.2               999.0            45.5      6.9              3.5            469.0                   1.4           461


**Insight:** aparece un segmento chico y claramente separado con **tiempos de entrega altos, más
demora y reseña promedio baja** (los "rezagados", candidatos a intervención prioritaria) frente a otros
con buen desempeño. Como el clustering incluye `reseña_promedio`, la separación se nota no solo en la
logística sino en la **satisfacción**, que es la métrica de negocio que más importa.

### Etiquetas de negocio para los segmentos

**Qué hacemos:** reemplazamos el número de cluster (que K-Means asigna de forma arbitraria) por una
**etiqueta de negocio** que se deriva del perfil de cada grupo: el de mayor % de demora es el de
*vendedores rezagados*, entre los otros dos el de más pedidos es el de *vendedores consolidados de
alto volumen* y el restante, el de *vendedores ágiles de bajo volumen*. También medimos qué parte de
los pedidos y de las demoras concentra cada segmento.

**Para qué:** la etiqueta se asigna por perfil y no por número, así que se mantiene aunque una nueva
corrida cambie la numeración interna de los clusters; y la participación en las demoras dimensiona
el peso real de cada segmento para el negocio.

In [13]:
ETIQ_REZAGADOS = "Rezagados"
ETIQ_CONSOLIDADOS = "Consolidados de alto volumen"
ETIQ_AGILES = "Ágiles de bajo volumen"

medias_cluster = perfil.groupby("cluster")[["pct_demorado", "pedidos"]].mean()
c_rezagados = medias_cluster["pct_demorado"].idxmax()
c_consolidados = medias_cluster.drop(index=c_rezagados)["pedidos"].idxmax()
c_agiles = [c for c in medias_cluster.index if c not in (c_rezagados, c_consolidados)][0]
ETIQUETAS_SEGMENTO = {c_agiles: ETIQ_AGILES, c_consolidados: ETIQ_CONSOLIDADOS,
                      c_rezagados: ETIQ_REZAGADOS}
ORDEN_SEGMENTOS = [ETIQ_AGILES, ETIQ_CONSOLIDADOS, ETIQ_REZAGADOS]

perfil["segmento"] = perfil["cluster"].map(ETIQUETAS_SEGMENTO)

# participación de cada segmento en ítems vendidos y en ítems demorados
items_seg = df[["seller_id", "envio_demorado", "price"]].merge(
    perfil["segmento"], left_on="seller_id", right_index=True, how="inner")
participacion = pd.DataFrame({
    "vendedores_%": perfil["segmento"].value_counts(normalize=True) * 100,
    "items_%": items_seg["segmento"].value_counts(normalize=True) * 100,
    "facturacion_%": items_seg.groupby("segmento")["price"].sum()
                     / items_seg["price"].sum() * 100,
    "items_demorados_%": items_seg.loc[items_seg["envio_demorado"] == 1, "segmento"]
                         .value_counts(normalize=True) * 100,
}).reindex(ORDEN_SEGMENTOS).round(1)

print("Perfil promedio por segmento:")
print(perfil.groupby("segmento")[FEATURES_CLUSTER].mean().reindex(ORDEN_SEGMENTOS).round(2).to_string())
print("\nParticipación de cada segmento (%):")
print(participacion.to_string())

Perfil promedio por segmento:
                              tiempo_entrega_promedio  pct_demorado  distancia_promedio  flete_promedio  pedidos  reseña_promedio  precio_promedio  categorias_distintas
segmento                                                                                                                                                                
Ágiles de bajo volumen                           9.50          0.04              519.56           18.74     6.62             4.25           116.42                  1.43
Consolidados de alto volumen                    11.77          0.08              594.84           20.11   107.11             4.05           130.82                  4.23
Rezagados                                       19.58          0.24              999.00           45.50     6.92             3.48           469.05                  1.44

Participación de cada segmento (%):
                              vendedores_%  items_%  facturacion_%  items_demorados_%
se

### Figuras del modelado para el informe

**Qué hacemos:** exportamos a `figuras_informe/` las figuras de evaluación del clasificador (curva ROC,
matriz de confusión normalizada, importancia de variables, distribución de probabilidades y captura de
reseñas negativas por percentil de riesgo) y las del clustering (codo + silhouette, perfil de los
segmentos y mapa de vendedores), todas con el mismo estilo sobrio del informe.

**Para qué:** el informe final y la presentación incrustan estas imágenes; generarlas acá garantiza que
siempre correspondan al modelo efectivamente entrenado.

In [14]:
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

FIG_INF_DIR = BASE / "figuras_informe"
FIG_INF_DIR.mkdir(exist_ok=True)

# estilo formal compartido con el notebook 12
AZUL = "#1F3A5F"; AZUL_MEDIO = "#4A6F94"; GRIS = "#8A939C"; GRIS_CLARO = "#D5DAE0"; ROJO = "#A23B3B"
COLOR_SEGMENTO = {ETIQ_AGILES: "#4A6F94", ETIQ_CONSOLIDADOS: "#1F3A5F", ETIQ_REZAGADOS: "#A23B3B"}
plt.rcParams.update({
    "font.family": "Arial", "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.edgecolor": "#5E666E", "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#E3E6EA", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "figure.facecolor": "white", "savefig.facecolor": "white", "legend.frameon": False,
})


def guardar_informe(nombre):
    ruta = FIG_INF_DIR / f"{nombre}.png"
    plt.savefig(ruta, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  -> {ruta.name}")


NOMBRES_VARIABLES = {
    "hist_resena_prom_vendedor": "Reseña histórica del vendedor",
    "distancia_km": "Distancia cliente-vendedor (km)",
    "freight_value": "Costo de flete (R$)",
    "mes_compra_num": "Mes de compra",
    "flete_ratio": "Flete / precio",
    "price": "Precio del producto (R$)",
    "product_weight_g": "Peso del producto (g)",
    "hist_pct_demora_vendedor": "% histórico de demora del vendedor",
    "volumen_cm3": "Volumen del producto (cm³)",
    "hist_pedidos_vendedor": "Pedidos históricos del vendedor",
    "items_por_pedido": "Ítems por pedido",
    "payment_installments_max": "Cuotas elegidas",
    "dia_semana": "Día de la semana",
    "hist_categorias_vendedor": "Categorías que vende el vendedor",
    "customer_state_RJ": "Cliente en Río de Janeiro",
}

print("Figuras del modelado:")

# 1) curva ROC
fpr, tpr, _ = roc_curve(y_test, proba)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.plot(fpr, tpr, color=AZUL, lw=2, label=f"LightGBM (AUC = {metricas['auc']:.2f})")
ax.plot([0, 1], [0, 1], ls="--", color=GRIS, lw=1, label="Clasificador al azar (AUC = 0,50)")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curva ROC del modelo de reseña (conjunto de prueba)")
ax.legend(loc="lower right")
guardar_informe("nb11_01_curva_roc")

# 2) matriz de confusión normalizada por fila
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.0, 4.2))
ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_norm[i, j]:.0%}\n({cm[i, j]:,})".replace(",", "."),
                ha="center", va="center", fontsize=11,
                color="white" if cm_norm[i, j] > 0.5 else "#1B1B1B")
ax.set_xticks([0, 1], ["Negativa", "Positiva"])
ax.set_yticks([0, 1], ["Negativa", "Positiva"])
ax.set_xlabel("Reseña predicha")
ax.set_ylabel("Reseña real")
ax.set_title("Matriz de confusión (porcentaje por clase real)")
guardar_informe("nb11_02_matriz_confusion")

# 3) importancia de variables
imp = top.rename(index=NOMBRES_VARIABLES).sort_values()
fig, ax = plt.subplots(figsize=(6.6, 4.8))
colores = [AZUL if "vendedor" in n.lower() else AZUL_MEDIO for n in imp.index]
ax.barh(imp.index, imp.values, color=colores)
ax.set_xlabel("Importancia (cantidad de divisiones en los árboles)")
ax.set_title("Las 15 variables más influyentes del modelo")
ax.grid(axis="y", visible=False)
guardar_informe("nb11_03_importancia_variables")

# 4) distribución de la probabilidad predicha según la clase real
fig, ax = plt.subplots(figsize=(6.6, 3.8))
bins = np.linspace(0, 1, 41)
ax.hist(proba[y_test.values == 1], bins=bins, density=True, alpha=0.75, color=AZUL_MEDIO,
        label="Reseña real positiva")
ax.hist(proba[y_test.values == 0], bins=bins, density=True, alpha=0.75, color=ROJO,
        label="Reseña real negativa")
ax.axvline(0.5, color="#1B1B1B", ls="--", lw=1)
ax.text(0.51, ax.get_ylim()[1] * 0.92, "umbral 0,5", fontsize=9)
ax.set_xlabel("Probabilidad estimada de reseña positiva")
ax.set_ylabel("Densidad")
ax.set_title("Separación de las clases según la probabilidad del modelo")
ax.legend(loc="upper left")
guardar_informe("nb11_04_distribucion_probabilidades")

# 5) captura acumulada de reseñas negativas al priorizar por riesgo
riesgo = 1 - proba
orden = np.argsort(-riesgo)
neg_ordenadas = (y_test.values[orden] == 0).astype(int)
captura = np.cumsum(neg_ordenadas) / neg_ordenadas.sum()
fraccion = np.arange(1, len(orden) + 1) / len(orden)
fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.plot(fraccion * 100, captura * 100, color=AZUL, lw=2, label="Priorizando por riesgo del modelo")
ax.plot([0, 100], [0, 100], ls="--", color=GRIS, lw=1, label="Selección al azar")
for p in (0.10, 0.20, 0.30):
    c = captura[int(p * len(orden)) - 1] * 100
    ax.scatter([p * 100], [c], color=ROJO, zorder=3, s=22)
    ax.annotate(f"{c:.0f}%", (p * 100, c), textcoords="offset points", xytext=(6, -12), fontsize=9)
ax.set_xlabel("% de pedidos revisados (de mayor a menor riesgo)")
ax.set_ylabel("% de reseñas negativas detectadas")
ax.set_title("Curva de captura de reseñas negativas")
ax.legend(loc="lower right")
guardar_informe("nb11_05_curva_captura")

for p in (0.10, 0.20, 0.30):
    print(f"     revisando el {p:.0%} de mayor riesgo se detecta el "
          f"{captura[int(p * len(orden)) - 1]:.1%} de las reseñas negativas")
tn, fp_, fn, tp = cm.ravel()
print(f"     recall negativas={tn / (tn + fp_):.3f} | precision negativas={tn / (tn + fn):.3f} | "
      f"recall positivas={tp / (tp + fn):.3f} | precision positivas={tp / (tp + fp_):.3f}")

# 6) codo + silhouette
fig, ax1 = plt.subplots(figsize=(6.6, 3.8))
ax1.plot(resultados_k["k"], resultados_k["inercia"], marker="o", color=AZUL, label="Inercia (codo)")
ax1.set_xlabel("Cantidad de segmentos (k)")
ax1.set_ylabel("Inercia", color=AZUL)
ax2 = ax1.twinx()
ax2.plot(resultados_k["k"], resultados_k["silhouette"], marker="s", color=ROJO, label="Silhouette")
ax2.set_ylabel("Silhouette", color=ROJO)
ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.axvline(K, color=GRIS, ls=":", lw=1)
ax1.set_title("Elección de la cantidad de segmentos")
lineas = ax1.get_lines()[:1] + ax2.get_lines()
ax1.legend(lineas, [l.get_label() for l in lineas], loc="upper right")
guardar_informe("nb11_06_codo_silhouette")

# 7) perfil comparativo de los segmentos
perfil_seg = perfil.groupby("segmento")[FEATURES_CLUSTER].mean().reindex(ORDEN_SEGMENTOS)
perfil_seg["pct_demorado"] = perfil_seg["pct_demorado"] * 100
paneles = [("tiempo_entrega_promedio", "Días de entrega promedio", "{:.1f}"),
           ("pct_demorado", "% de envíos demorados", "{:.1f}%"),
           ("reseña_promedio", "Reseña promedio (1-5)", "{:.2f}"),
           ("pedidos", "Pedidos por vendedor", "{:.0f}"),
           ("flete_promedio", "Flete promedio (R$)", "{:.1f}"),
           ("distancia_promedio", "Distancia promedio (km)", "{:.0f}")]
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.4))
etiquetas_cortas = ["Ágiles", "Consolidados", "Rezagados"]
for ax, (col, titulo, fmt) in zip(axes.ravel(), paneles):
    vals = perfil_seg[col].values
    ax.bar(etiquetas_cortas, vals, color=[COLOR_SEGMENTO[s] for s in ORDEN_SEGMENTOS])
    for x, v in enumerate(vals):
        ax.text(x, v, fmt.format(v), ha="center", va="bottom", fontsize=9)
    ax.set_title(titulo, fontsize=10)
    ax.set_ylim(0, vals.max() * 1.18)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="x", labelsize=9)
fig.suptitle("Perfil promedio de los tres segmentos de vendedores", fontweight="bold", fontsize=11)
fig.tight_layout()
guardar_informe("nb11_07_perfil_segmentos")

# 8) vendedores en el plano tiempo de entrega vs % de demora
fig, ax = plt.subplots(figsize=(6.8, 4.4))
for seg in ORDEN_SEGMENTOS:
    sub = perfil[perfil["segmento"] == seg]
    ax.scatter(sub["tiempo_entrega_promedio"], sub["pct_demorado"] * 100,
               s=np.clip(np.sqrt(sub["pedidos"]) * 4, 6, 120), alpha=0.45,
               color=COLOR_SEGMENTO[seg], edgecolors="none", label=f"{seg} ({len(sub):,})".replace(",", "."))
ax.set_xlim(0, 45)
ax.set_ylim(-2, 102)
ax.set_xlabel("Días de entrega promedio del vendedor")
ax.set_ylabel("% de envíos demorados")
ax.set_title("Vendedores según tiempo de entrega y demora (tamaño = pedidos)")
ax.legend(loc="upper right", markerscale=0.8)
guardar_informe("nb11_08_mapa_vendedores")

Figuras del modelado:


  -> nb11_01_curva_roc.png
  -> nb11_02_matriz_confusion.png


  -> nb11_03_importancia_variables.png


  -> nb11_04_distribucion_probabilidades.png


  -> nb11_05_curva_captura.png
     revisando el 10% de mayor riesgo se detecta el 30.6% de las reseñas negativas
     revisando el 20% de mayor riesgo se detecta el 46.5% de las reseñas negativas
     revisando el 30% de mayor riesgo se detecta el 57.4% de las reseñas negativas
     recall negativas=0.586 | precision negativas=0.330 | recall positivas=0.746 | precision positivas=0.894


  -> nb11_06_codo_silhouette.png


  -> nb11_07_perfil_segmentos.png


  -> nb11_08_mapa_vendedores.png


---
## Cierre del modelado

Los archivos que deja este notebook en `modelos_output/` alimentan el dashboard `app.py`:

| Archivo | Contenido |
|---|---|
| `modelo_final.joblib` | Clasificador entrenado + todo lo necesario para predecir compras nuevas |
| `catalogo_vendedores.csv` | Vendedores con historial real (para elegir en el dashboard) |
| `clasificacion_comparacion.csv` | Métricas del clasificador en test |
| `clasificacion_matriz_confusion.csv` | Matriz de confusión |
| `clasificacion_importancia.csv` | Top 15 variables por importancia |
| `clasificacion_predicciones_test.csv` | Predicciones y probabilidades sobre el test |
| `clustering_resumen.csv` | Perfil promedio de cada segmento de vendedores |
| `clustering_asignaciones.csv` | Cluster asignado a cada vendedor |
| `clustering_elbow_silhouette.csv` | Codo + silhouette para la elección de k |

Con esto, cualquiera puede reproducir el modelado ejecutando los notebooks **02 → 04 → 11** y luego
abrir el dashboard sin reentrenar nada.

In [15]:
print("Artefactos generados en modelos_output/:")
for f in sorted(OUT_DIR.glob('*')):
    print(f"  - {f.name}")

Artefactos generados en modelos_output/:
  - catalogo_vendedores.csv
  - clasificacion_comparacion.csv
  - clasificacion_importancia.csv
  - clasificacion_matriz_confusion.csv
  - clasificacion_predicciones_test.csv
  - clustering_asignaciones.csv
  - clustering_elbow_silhouette.csv
  - clustering_resumen.csv
  - modelo_final.joblib
